In [ ]:
%matplotlib widget

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import FloatSlider, HBox, HTML, Layout
from IPython.display import display

# ============================================================
# IMPULSE-INVARIANCE MAPPING: s-PLANE -> z-PLANE
# z = exp(s Ts)
# ============================================================

plt.ioff()

CONTENT_WIDTH = '900px'

plt.rcParams.update({'font.size':10.5,'axes.titlesize':12,'axes.labelsize':10.5,'xtick.labelsize':9,'ytick.labelsize':9,'legend.fontsize':8.5})

# ============================================================
# STYLE
# ============================================================

display(HTML("""
<style>

.ii-root{
    width:900px;
    max-width:900px;
    font-family:Arial,sans-serif;
}

.ii-header{
    background:linear-gradient(90deg,#1565c0,#1976d2);
    color:white;
    padding:11px 15px;
    border-radius:8px 8px 0 0;
    font-size:19px;
    font-weight:bold;
}

.ii-doc{
    background:#f6f9fd;
    border:1px solid #b9cce5;
    border-top:none;
    padding:11px 14px;
    border-radius:0 0 8px 8px;
    font-size:15px;
    line-height:1.55;
    margin-bottom:9px;
}

.ii-box{
    width:100%;
    box-sizing:border-box;
    border:1px solid #b9cce5;
    border-radius:7px;
    padding:10px 12px;
    margin-bottom:8px;
    font-size:14.5px;
    line-height:1.50;
}

.ii-title{
    font-weight:bold;
    color:#0d47a1;
    font-size:15.5px;
}

.ii-cols{
    display:flex;
    gap:18px;
    align-items:flex-start;
    flex-wrap:nowrap;
}

.ii-col{
    flex:1;
    min-width:0;
}

.widget-label{
    font-size:14px !important;
}

.jupyter-widgets input{
    font-size:13.5px !important;
}

.jupyter-widgets-output-area,
.widget-output,
.output_area,
.output_subarea,
.jp-OutputArea-output,
.jp-OutputArea-child{
    overflow-x:visible !important;
    overflow-y:visible !important;
    max-width:none !important;
}

.jp-OutputArea,
.output_wrapper,
.widget-box,
.jupyter-widgets{
    overflow:visible !important;
}

</style>
"""))

# ============================================================
# DOCUMENTATION
# ============================================================

display(HTML("""
<div class="ii-root">

<div class="ii-header">
Impulse-Invariance Mapping from the s-Plane to the z-Plane
</div>

<div class="ii-doc">

<b>Transformation.</b>
In the impulse-invariance method, an analog pole

<div style="text-align:center;font-size:16px;margin:8px 0;">
<b>s = σ + jΩ</b>
</div>

is mapped into the z-plane according to

<div style="text-align:center;font-size:16px;margin:8px 0;">
<b>z = e<sup>sT<sub>s</sub></sup>
= e<sup>σT<sub>s</sub></sup>
e<sup>jΩT<sub>s</sub></sup>.</b>
</div>

Therefore, <b>|z| = e<sup>σT<sub>s</sub></sup></b> and
<b>ω = ΩT<sub>s</sub></b> modulo 2π.

<br><br>

Consequently, the imaginary axis of the s-plane maps onto the unit circle,
the left half-plane maps inside the unit circle, and the right half-plane maps outside it.
The mapping of analog frequency onto digital frequency is many-to-one because frequencies
separated by integer multiples of 2π/T<sub>s</sub> map to the same digital frequency.

</div>

</div>
"""))

# ============================================================
# CONTROLS
# ============================================================

Ts_slider = FloatSlider(value=0.50,min=0.10,max=1.00,step=0.01,description='Ts:',continuous_update=True,readout_format='.2f',style={'description_width':'35px'},layout=Layout(width='220px'))

sigma_slider = FloatSlider(value=-0.50,min=-2.0,max=1.0,step=0.05,description='σ:',continuous_update=True,readout_format='.2f',style={'description_width':'35px'},layout=Layout(width='250px'))

Omega_slider = FloatSlider(value=2.0,min=-10.0,max=10.0,step=0.10,description='Ω:',continuous_update=True,readout_format='.2f',style={'description_width':'35px'},layout=Layout(width='280px'))

design_title = HTML('<div class="ii-title">Mapping parameters</div>',layout=Layout(width='140px'))

controls = HBox([
    design_title,
    Ts_slider,
    sigma_slider,
    Omega_slider
],layout=Layout(width=CONTENT_WIDTH,border='1px solid #b9cce5',padding='9px 12px',margin='0 0 8px 0',align_items='center'))

info = HTML(layout=Layout(width=CONTENT_WIDTH,margin='0 0 8px 0'))

# ============================================================
# FIGURE — CREATED ONCE
# ============================================================

fig,(ax_s,ax_z) = plt.subplots(1,2,figsize=(9.0,4.5))

fig.canvas.toolbar_visible = False
fig.canvas.header_visible = False
fig.canvas.footer_visible = False

# ============================================================
# s-PLANE
# ============================================================

ax_s.axhline(0,color='black',linewidth=0.8)
ax_s.axvline(0,color='black',linewidth=0.8)
ax_s.axvspan(-3,0,alpha=0.05,label='Stable half-plane')

imag_axis, = ax_s.plot([0,0],[-75,75],'--',linewidth=1.1,label=r'Imaginary axis $s=j\Omega$')

selected_s, = ax_s.plot([],[],'ro',markersize=7,label='Selected s-plane point')
alias_plus_s, = ax_s.plot([],[],'o',markersize=5,label=r'$\Omega+2\pi/T_s$')
alias_minus_s, = ax_s.plot([],[],'o',markersize=5,label=r'$\Omega-2\pi/T_s$')

ax_s.set_xlim(-2.5,1.5)
ax_s.set_ylim(-75,75)

ax_s.set_title('s-Plane')
ax_s.set_xlabel(r'$\Re\{s\}$')
ax_s.set_ylabel(r'$\Im\{s\}$')

ax_s.grid(True,linestyle=':',alpha=0.25)
ax_s.legend(loc='upper center',bbox_to_anchor=(0.5,-0.17),ncol=2,frameon=False)

# ============================================================
# z-PLANE
# ============================================================

theta = np.linspace(0,2*np.pi,1000)

unit_x = np.cos(theta)
unit_y = np.sin(theta)

ax_z.axhline(0,color='black',linewidth=0.8)
ax_z.axvline(0,color='black',linewidth=0.8)

ax_z.plot(unit_x,unit_y,'--',linewidth=1.2,label='Unit circle')

mapped_circle, = ax_z.plot([],[],color='red',linewidth=1.3,label=r'$|z|=e^{\sigma T_s}$')

selected_z, = ax_z.plot([],[],'ro',markersize=7,label='Mapped point')
alias_plus_z, = ax_z.plot([],[],'o',markersize=5,label='Alias image')
alias_minus_z, = ax_z.plot([],[],'o',markersize=5)

ax_z.set_xlim(-1.5,1.5)
ax_z.set_ylim(-1.5,1.5)
ax_z.set_aspect('equal',adjustable='box')

ax_z.set_title('z-Plane')
ax_z.set_xlabel(r'$\Re\{z\}$')
ax_z.set_ylabel(r'$\Im\{z\}$')

ax_z.grid(True,linestyle=':',alpha=0.25)
ax_z.legend(loc='upper center',bbox_to_anchor=(0.5,-0.17),ncol=2,frameon=False)

plt.subplots_adjust(left=0.08,right=0.98,top=0.92,bottom=0.25,wspace=0.28)

# ============================================================
# UPDATE
# ============================================================

def update_mapping(change=None):

    Ts = Ts_slider.value
    sigma = sigma_slider.value
    Omega = Omega_slider.value

    # --------------------------------------------------------
    # Selected analog point
    # --------------------------------------------------------

    s = sigma+1j*Omega

    # --------------------------------------------------------
    # Impulse-invariance mapping
    # --------------------------------------------------------

    z = np.exp(s*Ts)

    # --------------------------------------------------------
    # Alias frequencies
    # --------------------------------------------------------

    alias_spacing = 2*np.pi/Ts

    Omega_plus = Omega+alias_spacing
    Omega_minus = Omega-alias_spacing

    s_plus = sigma+1j*Omega_plus
    s_minus = sigma+1j*Omega_minus

    z_plus = np.exp(s_plus*Ts)
    z_minus = np.exp(s_minus*Ts)

    # --------------------------------------------------------
    # Radius corresponding to constant sigma
    # --------------------------------------------------------

    radius = np.exp(sigma*Ts)

    circle_x = radius*np.cos(theta)
    circle_y = radius*np.sin(theta)

    # --------------------------------------------------------
    # Update s-plane
    # --------------------------------------------------------

    selected_s.set_data([sigma],[Omega])
    alias_plus_s.set_data([sigma],[Omega_plus])
    alias_minus_s.set_data([sigma],[Omega_minus])

    # --------------------------------------------------------
    # Update z-plane
    # --------------------------------------------------------

    mapped_circle.set_data(circle_x,circle_y)

    selected_z.set_data([np.real(z)],[np.imag(z)])
    alias_plus_z.set_data([np.real(z_plus)],[np.imag(z_plus)])
    alias_minus_z.set_data([np.real(z_minus)],[np.imag(z_minus)])

    # --------------------------------------------------------
    # Numerical quantities
    # --------------------------------------------------------

    omega_raw = Omega*Ts
    omega_wrapped = (omega_raw+np.pi)%(2*np.pi)-np.pi

    stable_s = sigma < 0
    stable_z = np.abs(z) < 1

    info.value = f"""
    <div class="ii-root">

    <div class="ii-box">

    <div class="ii-title" style="margin-bottom:7px;">Current mapping</div>

    <div class="ii-cols">

    <div class="ii-col">
    Sampling period:<br>
    <b>T<sub>s</sub> = {Ts:.2f}</b><br><br>
    Analog point:<br>
    <b>s = {sigma:.2f} {Omega:+.2f}j</b>
    </div>

    <div class="ii-col">
    Mapped point:<br>
    <b>z = {np.real(z):.5f} {np.imag(z):+.5f}j</b><br><br>
    Radius:<br>
    <b>|z| = {np.abs(z):.5f}</b>
    </div>

    <div class="ii-col">
    Analog frequency:<br>
    <b>Ω = {Omega:.4f}</b><br><br>
    Digital frequency:<br>
    <b>ω = {omega_wrapped:.4f} rad</b>
    </div>

    <div class="ii-col">
    Alias spacing:<br>
    <b>2π/T<sub>s</sub> = {alias_spacing:.4f}</b><br><br>
    Stability:<br>
    <b>{"STABLE" if stable_s and stable_z else "UNSTABLE"}</b>
    </div>

    </div>

    </div>

    <div class="ii-box">

    <b>Many-to-one frequency mapping:</b><br><br>

    Ω = <b>{Omega:.4f}</b>,
    &nbsp;&nbsp;
    Ω + 2π/T<sub>s</sub> = <b>{Omega_plus:.4f}</b>,
    &nbsp;&nbsp;
    Ω - 2π/T<sub>s</sub> = <b>{Omega_minus:.4f}</b>

    <br><br>

    all map to the same digital point

    <b>
    z = {np.real(z):.5f} {np.imag(z):+.5f}j
    </b>.

    </div>

    </div>
    """

    fig.canvas.draw_idle()

# ============================================================
# EVENTS
# ============================================================

Ts_slider.observe(update_mapping,names='value')
sigma_slider.observe(update_mapping,names='value')
Omega_slider.observe(update_mapping,names='value')

# ============================================================
# DISPLAY
# ============================================================

display(controls)
display(info)
display(fig.canvas)

update_mapping()